In [8]:
"# Setup the Jupyter version of Dash\n",
from jupyter_dash import JupyterDash
from pymongo import MongoClient
from bson.json_util import dumps

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html, ctx, dash_table
import plotly.express as px
from dash.dependencies import Input, Output, State
import base64 # needed to import images

import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import re # needed for the regex pattern matching


from animal_shelter import AnimalShelter


##########################
#Data Manipulation / Model
##########################
username = 'aacuser'
password = 'greatPASSword1234'
host = '127.0.0.1'
port = 27017
db = 'AAC'
collection = 'animals'

db = AnimalShelter()
#db = AnimalShelter(username, password, host, port, database, collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.readRecord({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object to crash - so we remove it in the dataframe here. The df.drop
# command allows us to drop the column. If we do not set inplace=True - it will 
# return a new dataframe that contain the dropped columns
df.drop(columns=['_id'],inplace=True)

# DEBUG
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# add the company logo
image_file = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_file, 'rb').read())

app.layout = html.Div([
    # create anchor for the business logo
    # make the image a href to the website, www.snhu.edu
    # open link in a new tab be setting a blank target
    html.A([
        html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                            height=250, width = 251))], href = 'https://www.snhu.edu', target = "_blank"),
    html.Div(id='hidden-div', style={'display': 'none'}),
    html.Center(html.B(html.H1('KAskew SNHU CS-340 Dashboard'))),
    html.Hr(),
    # create the radio buttons to act as a filter
    # set the default on initial load to 'All'
    dcc.RadioItems(
        id = 'filter-type',
        options = [
            {'label': 'All', 'value': 'All'},
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster Rescue or Individual Tracking', 'value': 'Disaster'},
        ],
    value= 'All'
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                                 for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native", # allow filtering
                         sort_action="native",   # allow sorting
                         sort_mode="multi",
                         column_selectable=False,
                         row_deletable=False,
                         row_selectable='single', # allow row selection
                         selected_columns=[],
                         selected_rows=[0],
                         page_action="native",    # enable pagination
                         page_current=0,          # set start page
                         page_size=10             # set rows per page
                        ),
    html.Br(),
    html.Hr(),
    # set up the dashboard so that the chart and the geolcation are side-by-side
    html.Div(className='row',
             style={'display': 'flex', 'justify-content': 'center'},
             children = [
                 # Pie Chart information
                 html.Div(id='graph-id', className='col s12 m6'),
                 # geolocation information
                 html.Div(id='map-id', className='col s12 m6')
             ]
            )
])
    
#############################################
# Interaction Between Components / Controller
#############################################
# This method will handle dynamically generated components with pattern matching callbacks 
@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'columns')],
    [Input('filter-type', 'value')]
)
               
def update_dashboard(filter_type):
    # filter logic that sets df using if/elif statements
    if filter_type == 'All': 
        df = pd.DataFrame.from_records(db.readRecord({}))
    elif filter_type == 'Water':
        # implement cleaner searches by ignoring case inputs
        # build regex patterns for the different filter queries
        labRegex = re.compile(".*lab.*", re.IGNORECASE)
        chesaRegex = re.compile(".*chesa.*", re.IGNORECASE)
        newRegex = re.compile(".*new.*", re.IGNORECASE)
    
        df = pd.DataFrame.from_records(db.readRecord({
            '$or': [
                {"breed": {'$regex': newRegex}}, # pass the regex to the filter
                {"breed": {'$regex': chesaRegex}},
                {"breed": {'$regex': labRegex}},
            ],
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }))
    elif filter_type == 'Mountain':
        germanRegex = re.compile(".*german.*", re.IGNORECASE)
        alaskanRegex = re.compile(".*mala.*", re.IGNORECASE)
        oldRegex = re.compile(".*old english.*", re.IGNORECASE)
        huskyRegex = re.compile(".*husk.*", re.IGNORECASE)
        rottRegex = re.compile(".*rott.*", re.IGNORECASE)
    
        df = pd.DataFrame.from_records(db.readRecord({
            '$or': [
                {"breed": {'$regex': germanRegex}},
                {"breed": {'$regex': alaskanRegex}},
                {"breed": {'$regex': oldRegex}},
                {"breed": {'$regex': huskyRegex}},
                {"breed": {'$regex': rottRegex}},
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }))
    elif filter_type == 'Disaster':
        germanRegex = re.compile(".*german.*", re.IGNORECASE)
        goldenRegex = re.compile(".*golden.*", re.IGNORECASE)
        bloodRegex = re.compile(".*blood.*", re.IGNORECASE)
        doberRegex = re.compile(".*dober.*", re.IGNORECASE)
        rottRegex = re.compile(".*rott.*", re.IGNORECASE)
    
        df = pd.DataFrame.from_records(db.readRecord({
            '$or': [
                {"breed": {'$regex': germanRegex}},
                {"breed": {'$regex': goldenRegex}},
                {"breed": {'$regex': bloodRegex}},
                {"breed": {'$regex': doberRegex}},
                {"breed": {'$regex': rottRegex}},
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }))
    
    else:
        df = pd.DataFrame()
    
    # make results JSON safe and build columns each time the radio changes
    if '_id' in df.columns:
        df = df.drop(columns=['_id'])
    df = df.replace({np.nan: None})
    
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data = df.to_dict('records')
    
    # return order matches the outputs order
    return (data, columns)
    
# Display the breed of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    dffPie = pd.DataFrame(viewData)
    
    return [
        dcc.Graph(figure = px.pie(dffPie, names = 'breed')
        )
    ]

# This callback will highlight a row on the data table when the user selects it
@app.callback(Output('datatable-id', 'style_data_conditional'),
             [Input('datatable-id', 'selected_columns')]
             )
def update_styles(selected_columns):
    selected_columns = selected_columns or []
    return [{
        'if': { 'column_id': i },
        'backgroundColor': '#D2F3FF',
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable
# in the form of a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table
# in the form of a list. For this application, we are only permitting
# single row selection so there is only one calue in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)

def update_map(viewData, selected_rows):
    # Use current table view or the full df if not available
    dff = pd.DataFrame.from_dict(viewData) if viewData else df.copy()
    
    # Default to first row if nothing selected
    row = (selected_rows or [0])[0]
    row = max(0, min(row, len(dff) - 1)) # clamp in bounds
    
    # Prefer column names if present, else use template indices (13, 14)
    lat_col = 'location_lat' if 'location_lat' in dff.columns else None
    lon_col = 'location_long' if 'location_long' in dff.columns else None
    
    if lat_col and lon_col:
        lat = dff.iloc[row][lat_col]
        lon = dff.iloc[row][lon_col]
    else:
        # Fallback to positional indices
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]
        
        # Breed and Name columns
        breed = dff.iloc[row].get('breed', dff.iloc[row, 4] if dff.shape[1] > 4 else 'Not Found')
        name = dff.iloc[row].get('name', dff.iloc[row, 9] if dff.shape[1] > 9 else 'Not Found')
    
    # Austin TX is at [30.75, -97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   # Marker with tool tip and popup
                   # Column 13 and 14 define the grid-coordinates for
                   # the map
                   # Column 4 defines the breed for the animal
                   # Column 9 defines the name of the animal
                   dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                             children=[
                                 dl.Tooltip(dff.iloc[row,4]),
                                 dl.Popup([
                                     html.H1("Animal Name"),
                                     html.P(dff.iloc[row,9])
                                 ])
                             ])
               ])
    ]

# Ken Askew
app.run_server(debug=True)

Dash app running on http://127.0.0.1:11201/
